In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install timm grad-cam -q

import torch
print(torch.cuda.get_device_name(0))

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 44.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Tesla T4


In [ ]:
"""
MURA Shortcut Learning Project
Full pipeline: dataloader → train → evaluate (clean + masked) → GradCAM
"""

import os, glob, numpy as np, pandas as pd
from PIL import Image
from sklearn.metrics import roc_auc_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import timm

#Config
DATA_ROOT   = "/content/drive/MyDrive/MURA-v1.1"   # change if needed
BODY_PARTS  = ["XR_WRIST", "XR_ELBOW", "XR_SHOULDER"]
BATCH_SIZE  = 32
EPOCHS      = 20
LR          = 1e-4
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_DIR    = "/content/drive/MyDrive/mura_checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Using device: {DEVICE}")

#DATASET
class MURADataset(Dataset):
    """
    Loads images for a given split and list of body parts.
    Label is inferred from folder name: 'positive' = abnormal (1), 'negative' = normal (0).
    mask_artifacts=True blacks out corners and borders at test time.
    """
    def __init__(self, root, split, body_parts, transform=None, mask_artifacts=False):
        self.transform = transform
        self.mask_artifacts = mask_artifacts
        self.samples = []

        for bp in body_parts:
            bp_path = os.path.join(root, split, bp)
            if not os.path.exists(bp_path):
                print(f"Warning: {bp_path} not found, skipping.")
                continue
            for study_dir in glob.glob(os.path.join(bp_path, "*", "*")):
                label = 1 if "positive" in study_dir.lower() else 0
                for img_path in glob.glob(os.path.join(study_dir, "*.png")):
                    self.samples.append((img_path, label))

        print(f"[{split}] Loaded {len(self.samples)} images across {body_parts}")

    def __len__(self):
        return len(self.samples)

    def _apply_mask(self, img_tensor):
        """Black out corners (15%) and borders (10px) — artifact regions."""
        _, H, W = img_tensor.shape
        corner = int(0.15 * min(H, W))
        border = 10
        masked = img_tensor.clone()
        masked[:, :corner, :corner] = 0
        masked[:, :corner, -corner:] = 0
        masked[:, -corner:, :corner] = 0
        masked[:, -corner:, -corner:] = 0
        masked[:, :border, :] = 0
        masked[:, -border:, :] = 0
        masked[:, :, :border] = 0
        masked[:, :, -border:] = 0
        return masked

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        if self.mask_artifacts:
            img = self._apply_mask(img)
        return img, label, img_path

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


#MODELS
def get_model(name):
    """Return a pretrained model with the classifier head replaced for binary output."""
    if name == "resnet50":
        m = models.resnet50(weights="IMAGENET1K_V1")
        m.fc = nn.Linear(m.fc.in_features, 1)
    elif name == "efficientnet_b0":
        m = models.efficientnet_b0(weights="IMAGENET1K_V1")
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, 1)
    elif name == "vit":
        m = timm.create_model("vit_base_patch16_224", pretrained=True, num_classes=1)
    else:
        raise ValueError(f"Unknown model: {name}")
    return m.to(DEVICE)


#TRAINING LOOP
def train_model(model, train_loader, val_loader, model_name, body_part):
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    criterion = nn.BCEWithLogitsLoss()
    best_auc = 0

    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0
        for imgs, labels, _ in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.float().unsqueeze(1).to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        scheduler.step()

        auc = evaluate(model, val_loader)
        print(f"[{model_name} | {body_part}] Epoch {epoch+1}/{EPOCHS}  "
              f"Loss: {running_loss/len(train_loader):.4f}  Val AUC: {auc:.4f}")

        if auc > best_auc:
            best_auc = auc
            ckpt_path = os.path.join(SAVE_DIR, f"{model_name}_{body_part}_best.pt")
            torch.save(model.state_dict(), ckpt_path)

    print(f"Best AUC for {model_name} on {body_part}: {best_auc:.4f}")
    return best_auc


#EVALUATION
def evaluate(model, loader):
    model.eval()
    all_labels, all_preds = [], []
    with torch.no_grad():
        for imgs, labels, _ in loader:
            imgs = imgs.to(DEVICE)
            logits = model(imgs).squeeze(1).cpu()
            preds = torch.sigmoid(logits).numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    if len(set(all_labels)) < 2:
        return 0.5  # can't compute AUC with one class
    return roc_auc_score(all_labels, all_preds)


#ENSEMBLE EVALUATION
def evaluate_ensemble(models_dict, loader, weights):
    """
    models_dict: {"resnet50": model, "efficientnet_b0": model, ...}
    weights:     {"resnet50": 0.4, ...}  — from individual val AUCs
    """
    for m in models_dict.values():
        m.eval()

    all_labels, all_preds = [], []
    with torch.no_grad():
        for imgs, labels, _ in loader:
            imgs = imgs.to(DEVICE)
            weighted_sum = torch.zeros(imgs.size(0))
            total_weight = sum(weights.values())
            for name, model in models_dict.items():
                logits = model(imgs).squeeze(1).cpu()
                probs = torch.sigmoid(logits)
                weighted_sum += (weights[name] / total_weight) * probs
            all_preds.extend(weighted_sum.numpy())
            all_labels.extend(labels.numpy())

    return roc_auc_score(all_labels, all_preds)


#GRADCAM
def generate_gradcam(model, model_name, loader, save_dir, n_images=5):
    """Save GradCAM heatmap PNGs for the first n_images in loader."""
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.image import show_cam_on_image
    import cv2

    os.makedirs(save_dir, exist_ok=True)

    if model_name == "resnet50":
        target_layers = [model.layer4[-1]]
    elif model_name == "efficientnet_b0":
        target_layers = [model.features[-1]]
    elif model_name == "vit":
        target_layers = [model.blocks[-1].norm1]
    else:
        return

    cam = GradCAM(model=model, target_layers=target_layers)
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])

    count = 0
    for imgs, labels, paths in loader:
        for i in range(imgs.size(0)):
            if count >= n_images:
                break
            img_tensor = imgs[i].unsqueeze(0).to(DEVICE)
            grayscale_cam = cam(input_tensor=img_tensor)[0]

            img_np = imgs[i].permute(1,2,0).numpy()
            img_np = (img_np * std + mean).clip(0, 1).astype(np.float32)

            visualization = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)
            label_str = "abnormal" if labels[i].item() == 1 else "normal"
            out_path = os.path.join(save_dir, f"{model_name}_{label_str}_{count}.png")
            cv2.imwrite(out_path, cv2.cvtColor(visualization, cv2.COLOR_RGB2BGR))
            count += 1
        if count >= n_images:
            break

    print(f"Saved {count} GradCAM images to {save_dir}")


#MAIN
if __name__ == "__main__":
    results = {}

    for bp in BODY_PARTS:
        print(f"\n{'='*60}\nBody part: {bp}\n{'='*60}")
        results[bp] = {}

        train_ds = MURADataset(DATA_ROOT, "train", [bp], transform=train_transform)
        val_ds   = MURADataset(DATA_ROOT, "valid", [bp], transform=val_transform)
        val_masked_ds = MURADataset(DATA_ROOT, "valid", [bp], transform=val_transform, mask_artifacts=True)

        train_loader  = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
        val_loader    = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
        val_masked_loader = DataLoader(val_masked_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

        val_aucs = {}
        trained_models = {}

        for model_name in ["resnet50", "efficientnet_b0", "vit"]:
            print(f"\n--- Training {model_name} ---")
            model = get_model(model_name)
            best_auc = train_model(model, train_loader, val_loader, model_name, bp)

            ckpt = os.path.join(SAVE_DIR, f"{model_name}_{bp}_best.pt")
            model.load_state_dict(torch.load(ckpt))

            clean_auc  = evaluate(model, val_loader)
            masked_auc = evaluate(model, val_masked_loader)
            delta_auc  = clean_auc - masked_auc

            results[bp][model_name] = {
                "clean_auc":  round(clean_auc,  4),
                "masked_auc": round(masked_auc, 4),
                "delta_auc":  round(delta_auc,  4),
            }
            val_aucs[model_name]    = clean_auc
            trained_models[model_name] = model

            print(f"{model_name} | Clean AUC: {clean_auc:.4f} | Masked AUC: {masked_auc:.4f} | ΔAUC: {delta_auc:.4f}")

            gradcam_dir = f"/content/drive/MyDrive/gradcam/{bp}/{model_name}"
            generate_gradcam(model, model_name, val_loader, gradcam_dir, n_images=5)

        ensemble_configs = {
            "resnet50+efficientnet_b0":       ["resnet50", "efficientnet_b0"],
            "resnet50+vit":                   ["resnet50", "vit"],
            "efficientnet_b0+vit":            ["efficientnet_b0", "vit"],
            "resnet50+efficientnet_b0+vit":   ["resnet50", "efficientnet_b0", "vit"],
        }

        for ens_name, members in ensemble_configs.items():
            ens_models  = {m: trained_models[m] for m in members}
            ens_weights = {m: val_aucs[m] for m in members}

            clean_auc  = evaluate_ensemble(ens_models, val_loader,        ens_weights)
            masked_auc = evaluate_ensemble(ens_models, val_masked_loader,  ens_weights)
            delta_auc  = clean_auc - masked_auc

            results[bp][ens_name] = {
                "clean_auc":  round(clean_auc,  4),
                "masked_auc": round(masked_auc, 4),
                "delta_auc":  round(delta_auc,  4),
            }
            print(f"{ens_name} | Clean: {clean_auc:.4f} | Masked: {masked_auc:.4f} | ΔAUC: {delta_auc:.4f}")

    print("\n\n===== FINAL RESULTS =====")
    print(f"{'Model':<35} {'Part':<12} {'Clean AUC':>10} {'Masked AUC':>11} {'ΔAUC':>7}")
    print("-" * 78)
    for bp, models_res in results.items():
        for model_name, metrics in models_res.items():
            print(f"{model_name:<35} {bp:<12} "
                  f"{metrics['clean_auc']:>10.4f} {metrics['masked_auc']:>11.4f} "
                  f"{metrics['delta_auc']:>7.4f}")

    rows = []
    for bp, models_res in results.items():
        for model_name, metrics in models_res.items():
            rows.append({"body_part": bp, "model": model_name, **metrics})
    df = pd.DataFrame(rows)
    csv_path = "/content/drive/MyDrive/mura_results.csv"
    df.to_csv(csv_path, index=False)
    print(f"\nResults saved to {csv_path}")

Using device: cuda

Body part: XR_WRIST
[train] Loaded 0 images across ['XR_WRIST']
[valid] Loaded 0 images across ['XR_WRIST']
[valid] Loaded 0 images across ['XR_WRIST']


ValueError: num_samples should be a positive integer value, but got num_samples=0

In [ ]:
def generate_gradcam(model, model_name, loader, save_dir, n_images=5):
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.image import show_cam_on_image
    import cv2

    os.makedirs(save_dir, exist_ok=True)

    if model_name == "resnet50":
        target_layers = [model.layer4[-1]]
    elif model_name == "efficientnet_b0":
        target_layers = [model.features[-1]]
    elif model_name == "vit":
        target_layers = [model.blocks[-1].norm1]
    else:
        return

    def reshape_transform(tensor, height=14, width=14):
        result = tensor[:, 1:, :].reshape(tensor.size(0), height, width, tensor.size(2))
        return result.transpose(2, 3).transpose(1, 2)

    if model_name == "vit":
        cam = GradCAM(model=model, target_layers=target_layers,
                      reshape_transform=reshape_transform)
    else:
        cam = GradCAM(model=model, target_layers=target_layers)

    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])

    count = 0
    for imgs, labels, paths in loader:
        for i in range(imgs.size(0)):
            if count >= n_images:
                break
            img_tensor = imgs[i].unsqueeze(0).to(DEVICE)
            grayscale_cam = cam(input_tensor=img_tensor)[0]
            img_np = imgs[i].permute(1,2,0).numpy()
            img_np = (img_np * std + mean).clip(0, 1).astype(np.float32)
            visualization = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)
            label_str = "abnormal" if labels[i].item() == 1 else "normal"
            out_path = os.path.join(save_dir, f"{model_name}_{label_str}_{count}.png")
            cv2.imwrite(out_path, cv2.cvtColor(visualization, cv2.COLOR_RGB2BGR))
            count += 1
        if count >= n_images:
            break
    print(f"Saved {count} GradCAM images to {save_dir}")

In [ ]:
gradcam_dir = f"/content/drive/MyDrive/gradcam/XR_WRIST/vit"
generate_gradcam(trained_models["vit"], "vit", val_loader, gradcam_dir, n_images=5)

In [ ]:
ensemble_configs = {
    "resnet50+efficientnet_b0":       ["resnet50", "efficientnet_b0"],
    "resnet50+vit":                   ["resnet50", "vit"],
    "efficientnet_b0+vit":            ["efficientnet_b0", "vit"],
    "resnet50+efficientnet_b0+vit":   ["resnet50", "efficientnet_b0", "vit"],
}

for ens_name, members in ensemble_configs.items():
    ens_models  = {m: trained_models[m] for m in members}
    ens_weights = {m: val_aucs[m] for m in members}

    clean_auc  = evaluate_ensemble(ens_models, val_loader, ens_weights)
    masked_auc = evaluate_ensemble(ens_models, val_masked_loader, ens_weights)
    delta_auc  = clean_auc - masked_auc

    results["XR_WRIST"][ens_name] = {
        "clean_auc":  round(clean_auc,  4),
        "masked_auc": round(masked_auc, 4),
        "delta_auc":  round(delta_auc,  4),
    }
    print(f"{ens_name} | Clean: {clean_auc:.4f} | Masked: {masked_auc:.4f} | ΔAUC: {delta_auc:.4f}")

In [ ]:
for bp in ["XR_ELBOW", "XR_SHOULDER"]:
    results[bp] = {}
    train_ds = MURADataset(DATA_ROOT, "train", [bp], transform=train_transform)
    val_ds = MURADataset(DATA_ROOT, "valid", [bp], transform=val_transform)
    val_masked_ds = MURADataset(DATA_ROOT, "valid", [bp], transform=val_transform, mask_artifacts=True)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    val_masked_loader = DataLoader(val_masked_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    val_aucs = {}
    trained_models = {}

    for model_name in ["resnet50", "efficientnet_b0", "vit"]:
        print(f"\n--- Training {model_name} on {bp} ---")
        model = get_model(model_name)
        best_auc = train_model(model, train_loader, val_loader, model_name, bp)
        ckpt = os.path.join(SAVE_DIR, f"{model_name}_{bp}_best.pt")
        model.load_state_dict(torch.load(ckpt))
        clean_auc = evaluate(model, val_loader)
        masked_auc = evaluate(model, val_masked_loader)
        delta_auc = clean_auc - masked_auc
        results[bp][model_name] = {
            "clean_auc": round(clean_auc, 4),
            "masked_auc": round(masked_auc, 4),
            "delta_auc": round(delta_auc, 4),
        }
        val_aucs[model_name] = clean_auc
        trained_models[model_name] = model
        print(f"{model_name} | Clean: {clean_auc:.4f} | Masked: {masked_auc:.4f} | ΔAUC: {delta_auc:.4f}")
        gradcam_dir = f"/content/drive/MyDrive/gradcam/{bp}/{model_name}"
        generate_gradcam(model, model_name, val_loader, gradcam_dir, n_images=5)

    for ens_name, members in ensemble_configs.items():
        ens_models = {m: trained_models[m] for m in members}
        ens_weights = {m: val_aucs[m] for m in members}
        clean_auc = evaluate_ensemble(ens_models, val_loader, ens_weights)
        masked_auc = evaluate_ensemble(ens_models, val_masked_loader, ens_weights)
        delta_auc = clean_auc - masked_auc
        results[bp][ens_name] = {
            "clean_auc": round(clean_auc, 4),
            "masked_auc": round(masked_auc, 4),
            "delta_auc": round(delta_auc, 4),
        }
        print(f"{ens_name} | Clean: {clean_auc:.4f} | Masked: {masked_auc:.4f} | ΔAUC: {delta_auc:.4f}")

import pandas as pd
rows = []
for bp, models_res in results.items():
    for model_name, metrics in models_res.items():
        rows.append({"body_part": bp, "model": model_name, **metrics})
df = pd.DataFrame(rows)
df.to_csv("/content/drive/MyDrive/mura_results.csv", index=False)
print("\nDone! Results saved to mura_results.csv")